In [2]:
import pandas as pd
import requests

/Users/sophia/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
url = "https://data.cdc.gov/resource/pwn4-m3yp.json"
count_response = requests.get(url, params={"$select": "count(*)"})
print(count_response.json())
response = requests.get(url, params = {"$limit": 10380, "$order": "state, date_updated"})
covid = pd.DataFrame(response.json())

[{'count': '10380'}]


In [ ]:
# 1. Confirm actual column names returned by the API
print(covid.columns.tolist())


['date_updated', 'state', 'start_date', 'end_date', 'tot_cases', 'new_cases', 'tot_deaths', 'new_deaths', 'new_historic_cases', 'new_historic_deaths']


In [ ]:
# 2. Check actual data volume per state, not just the sum to determine why case sums look so low

diagnostic = covid.groupby('state')['new_deaths'].agg(['sum', 'count', 'size'])
print(diagnostic)

                                                     sum  count  size
state                                                                
AK     0.00.00.00.00.00.00.00.00.01.02.04.02.00.00.01...    173   173
AL     0.00.00.00.00.00.00.00.00.01.025.039.045.083.0...    173   173
AR     0.00.00.00.00.00.00.00.03.016.012.019.016.034....    173   173
AS     0.00.00.00.00.00.00.00.00.00.00.00.00.00.00.00...    173   173
AZ     0.00.00.00.00.00.00.00.00.06.026.057.066.076.0...    173   173
CA     0.00.00.00.00.00.01.02.012.049.0108.0270.0346....    173   173
CO     0.00.00.00.00.00.00.00.05.021.060.0147.0230.02...    173   173
CT     0.00.00.00.00.00.00.00.01.018.066.0241.0542.06...    173   173
DC     0.00.00.00.00.00.00.00.00.03.08.016.045.055.07...    173   173
DE     0.00.00.00.00.00.00.00.00.02.011.019.035.052.0...    173   173
FL     0.00.00.00.00.00.00.02.05.016.064.0222.0287.02...    173   173
FSM    0.00.00.00.00.00.00.00.00.00.00.00.00.00.00.00...    173   173
GA     0.00.00.00.00

In [ ]:
# 3. Confirm new_historic_cases/new_historic_deaths are sparse
# not real weekly counts - most values are 0 by design
print(covid[['state', 'date_updated', 'new_historic_cases', 'new_historic_deaths']].head(20))


   state             date_updated new_historic_cases new_historic_deaths
0     AK  2020-01-23T00:00:00.000                  0                   0
1     AK  2020-01-30T00:00:00.000                  0                   0
2     AK  2020-02-06T00:00:00.000                  0                   0
3     AK  2020-02-13T00:00:00.000                  0                   0
4     AK  2020-02-20T00:00:00.000                  0                   0
5     AK  2020-02-27T00:00:00.000                  0                   0
6     AK  2020-03-05T00:00:00.000                  0                   0
7     AK  2020-03-12T00:00:00.000                  0                   0
8     AK  2020-03-19T00:00:00.000                  0                   0
9     AK  2020-03-26T00:00:00.000                  0                   0
10    AK  2020-04-02T00:00:00.000                  0                   0
11    AK  2020-04-09T00:00:00.000                  0                   0
12    AK  2020-04-16T00:00:00.000                  

In [ ]:
# 4. Check for a single week reconciliation spike (example: FL)
fl = covid[covid['state'] == 'FL'][['date_updated', 'new_historic_cases', 'new_historic_deaths']].sort_values('date_updated')
print(fl.to_string())

last_week = fl.tail(3)
print("\nLast 3 reporting weeks in FL:")
print(last_week)

                 date_updated new_historic_cases new_historic_deaths
1730  2020-01-23T00:00:00.000                  0                   0
1731  2020-01-30T00:00:00.000                  0                   0
1732  2020-02-06T00:00:00.000                  0                   0
1733  2020-02-13T00:00:00.000                  0                   0
1734  2020-02-20T00:00:00.000                  0                   0
1735  2020-02-27T00:00:00.000                  0                   0
1736  2020-03-05T00:00:00.000                  0                   0
1737  2020-03-12T00:00:00.000                  0                   0
1738  2020-03-19T00:00:00.000                  0                   0
1739  2020-03-26T00:00:00.000                  0                   0
1740  2020-04-02T00:00:00.000                  0                   0
1741  2020-04-09T00:00:00.000                  0                   0
1742  2020-04-16T00:00:00.000                  0                   0
1743  2020-04-23T00:00:00.000     

In [12]:
# Confirm if this spike lands on the same final date for all states
print(covid.groupby('state')['date_updated'].max().unique())

['2023-05-11T00:00:00.000']
